# Actividad 2.1 – Estandarización de Pipelines y Control de Calidad de Datos

Este notebook continúa la Actividad 1.1. El experimento de clasificación de Iris con Random Forest y MLflow
pasa de ejecutarse celda por celda a ser un **pipeline único** con cinco etapas que se ejecutan con un solo comando y que se detienen si los datos o el modelo no cumplen los controles de calidad.

El notebook tiene dos partes:

1. **Construcción del repositorio.** Cada archivo del proyecto aparece en una celda `%%writefile` precedida
   de su explicación. Al ejecutarlas, el archivo se guarda en disco con exactamente el contenido mostrado.
2. **Ejecución y evidencias.** Se corre el pipeline completo, se revisan los reportes, se demuestra qué pasa
   cuando llegan datos inválidos, se prueba el contrato de datos y se consultan los runs en MLflow.


IMPORTANTE: El notebook debe ejecutarse desde la carpeta raíz del repositorio.

## 1. Entorno y estructura de carpetas

Las dependencias tienen versiones fijas en `requirements.txt` (sección 11). Para instalarlas en un entorno nuevo, se activa el entorno virtual y en la terminal se ejecuta: `python -m pip install -r requirements.txt`.

In [1]:
import json
import os
import platform
import subprocess
import sys
from pathlib import Path

import pandas as pd
import sklearn

os.environ["MLFLOW_DISABLE_AGENT_HINT"] = "1"   # silencia un aviso informativo de
                                                # MLflow
os.environ["GIT_PYTHON_REFRESH"] = "quiet"      # MLflow no necesita Git en esta
                                                # actividad
import mlflow

RAIZ = Path.cwd()
for carpeta in ["config", "contracts", "src", "tests"]:
    (RAIZ / carpeta).mkdir(exist_ok=True)

print("Python      :", platform.python_version())
print("scikit-learn:", sklearn.__version__)
print("MLflow      :", mlflow.__version__)
print("Carpeta     :", RAIZ)

Python      : 3.12.10
scikit-learn: 1.9.1
MLflow      : 3.18.0
Carpeta     : d:\1. Esp. Ciencia de datos\2026-2\6. Seminario de grado 2\2. Segunda entrega\Repositorio_Entrega2


## 2. Configuración del experimento - `config/params.json`

En un solo archivo se consolidan los parámetros que estaban distribuidos en el notebook de la Actividad 1.1, como las semillas, la proporción de prueba, los hiperparámetros de la baseline y del modelo seleccionado, los umbrales mínimos de calidad y la conexión con MLflow. De esta manera, si se desea probar otra configuración, basta con modificar este archivo sin necesidad de cambiar el código.

- `split_seed` y `test_size` mantienen la semilla y la proporción de la Actividad
1.1 (120 flores para entrenamiento y 30 para prueba). La diferencia es que ahora la partición es estratificada, por lo que cada especie conserva su proporción en ambos conjuntos.
- Se mantiene el modelo Run_2_n120_depth4, seleccionado en la Actividad 1.1 porque obtuvo resultados equivalentes a la baseline sin memorizar los datos de entrenamiento. La baseline se conserva como punto de referencia para comparar su desempeño.
- Los umbrales de 0.90 establecen el mínimo de calidad que debe cumplir el modelo para que el pipeline lo apruebe.

In [2]:
%%writefile config/params.json
{
  "datos": {
    "test_size": 0.2,
    "split_seed": 42,
    "estratificar": true
  },
  "entrenamiento": {
    "model_seed": 42,
    "cv_folds": 5
  },
  "baseline": {
    "run_name": "Pipeline_Baseline",
    "n_estimators": 100,
    "max_depth": null
  },
  "modelo_seleccionado": {
    "run_name": "Pipeline_Run_2_n120_depth4",
    "n_estimators": 120,
    "max_depth": 4
  },
  "umbrales_calidad": {
    "accuracy_min": 0.9,
    "f1_macro_min": 0.9,
    "cv_accuracy_min": 0.9
  },
  "mlflow": {
    "tracking_uri": "sqlite:///mlflow.db",
    "experimento": "Proyecto_Final_Grado"
  }
}

Overwriting config/params.json


## 3. Contrato de datos - `contracts/data_contract.json`

El archivo `contracts/data_contract.json` reúne las reglas que deben cumplir los datos que recibe el modelo y los resultados que entrega, pensando en su futura implementación para inferencia.

- **Campos de entrada.** Se definen las cuatro medidas de las flores, su tipo de dato, unidad, si son obligatorias, si aceptan nulos y sus valores mínimo y máximo. Se conservan los mismos nombres de la API de la Semana 3 del portal MLOps Santoto.
- **Rangos lógicos.** Se toman los valores de las 150 flores para establecer los límites permitidos, ampliándolos al múltiplo de 0,5 cm más cercano y sin bajar de
0,1 cm. Así se evita que el modelo reciba medidas demasiado alejadas de las que conoce.
- **Regla entre campos.** Se verifica que la longitud del pétalo sea menor que la del sépalo, ya que esto se cumple en todas las flores del dataset. Si no se cumple, podría ser señal de que las columnas se ingresaron equivocadamente.
- **Reglas de conjunto.** Los archivos de entrenamiento y prueba deben contener las tres especies, y ninguna flor de prueba puede aparecer también en el entrenamiento.
- **Variable objetivo.** `species` se utiliza únicamente en el entrenamiento y la evaluación, y no debe enviarse al momento de predecir, pues esto implicaría darle al modelo información que debe calcular.
- **Ejemplos.** Se agregan ejemplos de datos correctos e incorrectos para comprobar que las reglas funcionen. El ejemplo incorrecto incluye un error de cada tipo como lo son un texto en lugar de un número, un valor nulo, un valor fuera de los límites y el envío de la variable `species`, con sus respectivos mensajes.

El validador de la sección 6 utiliza este archivo como referencia, por lo que las mismas reglas se aplican tanto al entrenamiento como a la inferencia.

In [3]:
%%writefile contracts/data_contract.json
{
  "nombre": "contrato_inferencia_iris",
  "version": "1.0.0",
  "fecha": "2026-11-18",
  "responsable": "Laura Tatiana Rincón López",
  "descripcion": "Contrato oficial de datos para la inferencia del clasificador de especies Iris. Define qué debe recibir el modelo, en qué formato y dentro de qué límites, y qué devuelve. Lo usan la etapa de validación del pipeline (src/validate_data.py), el script de inferencia (src/predict.py) y la futura API de la Semana 3.",
  "modelo": {
    "algoritmo": "RandomForestClassifier (scikit-learn)",
    "configuracion": "Pipeline_Run_2_n120_depth4 (n_estimators=120, max_depth=4, random_state=42)",
    "artefacto": "models/model.pkl",
    "orden_caracteristicas": ["sepal_length", "sepal_width", "petal_length", "petal_width"]
  },
  "entrada": {
    "formato": "Objeto JSON con exactamente las cuatro características. También se acepta una lista de objetos para inferencia por lotes.",
    "campos_adicionales_permitidos": false,
    "campos": {
      "sepal_length": {
        "tipo": "float",
        "unidad": "cm",
        "requerido": true,
        "admite_nulos": false,
        "minimo": 4.0,
        "maximo": 8.0,
        "descripcion": "Longitud del sépalo"
      },
      "sepal_width": {
        "tipo": "float",
        "unidad": "cm",
        "requerido": true,
        "admite_nulos": false,
        "minimo": 2.0,
        "maximo": 4.5,
        "descripcion": "Ancho del sépalo"
      },
      "petal_length": {
        "tipo": "float",
        "unidad": "cm",
        "requerido": true,
        "admite_nulos": false,
        "minimo": 1.0,
        "maximo": 7.0,
        "descripcion": "Longitud del pétalo"
      },
      "petal_width": {
        "tipo": "float",
        "unidad": "cm",
        "requerido": true,
        "admite_nulos": false,
        "minimo": 0.1,
        "maximo": 2.5,
        "descripcion": "Ancho del pétalo"
      }
    },
    "reglas_entre_campos": [
      {
        "id": "petalo_menor_que_sepalo",
        "campo_menor": "petal_length",
        "campo_mayor": "sepal_length",
        "descripcion": "En las 150 flores del dataset la longitud del pétalo es menor que la del sépalo. Un registro que no cumpla esta relación probablemente tiene las columnas intercambiadas."
      }
    ],
    "criterio_rangos": "Rango observado en las 150 flores del dataset Iris, ampliado al múltiplo de 0,5 cm más cercano y sin bajar de 0,1 cm, porque una medida física debe ser positiva. Fuera de este rango el modelo extrapolaría sobre valores que nunca vio durante el entrenamiento."
  },
  "reglas_de_conjunto": [
    {
      "id": "clases_completas",
      "descripcion": "Los archivos de entrenamiento y prueba deben contener las tres especies. Si falta una, el modelo no puede aprenderla o no se puede evaluar."
    },
    {
      "id": "sin_solapamiento_train_test",
      "descripcion": "Ninguna flor de prueba puede estar también en entrenamiento. Si lo está, el modelo no predice sino que recuerda (fuga de datos). Solo se tolera la flor que ya viene duplicada en el dataset original."
    }
  ],
  "variable_objetivo": {
    "nombre": "species",
    "tipo": "int",
    "valores_permitidos": [0, 1, 2],
    "etiquetas": {"0": "setosa", "1": "versicolor", "2": "virginica"},
    "uso": "Solo existe en los archivos de entrenamiento y prueba. Nunca debe enviarse en una solicitud de inferencia, ya que el modelo la debe predecir y recibirla sería una fuga de datos."
  },
  "salida": {
    "formato": "Objeto JSON por cada registro recibido",
    "campos": {
      "prediction": {"tipo": "string", "valores_posibles": ["setosa", "versicolor", "virginica"]},
      "class_id": {"tipo": "int", "valores_posibles": [0, 1, 2]}
    }
  },
  "manejo_de_errores": "Si un registro incumple cualquier regla, no se predice: se devuelve la lista de errores con el campo, la regla incumplida y el valor recibido.",
  "ejemplo_entrada_valida": {
    "sepal_length": 6.0,
    "sepal_width": 2.9,
    "petal_length": 4.5,
    "petal_width": 1.4
  },
  "ejemplo_salida_valida": {
    "prediction": "versicolor",
    "class_id": 1
  },
  "ejemplo_entrada_invalida": {
    "sepal_length": "5.1",
    "sepal_width": null,
    "petal_length": -1.4,
    "petal_width": 0.2,
    "species": 0
  },
  "errores_esperados_entrada_invalida": [
    {"campo": "sepal_length", "regla": "tipo", "motivo": "Llega como texto (\"5.1\") y el contrato exige un número."},
    {"campo": "sepal_width", "regla": "nulo", "motivo": "El valor es null y el campo no admite nulos."},
    {"campo": "petal_length", "regla": "rango", "motivo": "-1.4 cm está por debajo del mínimo de 1.0 cm. Una longitud no puede ser negativa."},
    {"campo": "species", "regla": "columnas", "motivo": "Campo no permitido: es la variable objetivo y enviarla en la inferencia sería fuga de datos."}
  ]
}

Overwriting contracts/data_contract.json


## 4. Utilidades compartidas - `src/utils.py`

En `src/utils.py` se centralizan los elementos que se utilizan en diferentes partes del proyecto, como las rutas de los archivos, la lectura de la configuración y del contrato de datos, el cálculo del hash SHA-256, la consulta de las versiones del entorno y la configuración de los logs.

Todas las etapas registran su actividad en el mismo archivo `logs/pipeline.log`, lo que permite llevar un seguimiento de las validaciones aprobadas y fallidas y facilita su revisión.

El hash SHA-256 funciona como una huella digital del archivo dado que basta con modificar un solo valor para que el hash sea diferente. Esto permite comprobar que `train.csv` no cambió entre la validación y el entrenamiento.

In [4]:
%%writefile src/__init__.py
#Paquete del pipeline de clasificación de Iris.

Overwriting src/__init__.py


In [5]:
%%writefile src/utils.py
"""Rutas, carga de configuración y registro (log) compartidos por todas las etapas.

Al centralizar estos elementos, se evita que cada script maneje sus propias rutas o
registros de log. Así, todos trabajan con la misma configuración y guardan sus
registros en un único archivo `logs/pipeline.log`, facilitando el seguimiento de todo
el proceso.
"""

import hashlib
import json
import logging
import os
import platform
from pathlib import Path

# Raíz del repositorio: carpeta que contiene a src/
RAIZ = Path(__file__).resolve().parent.parent

RUTA_PARAMS = RAIZ / "config" / "params.json"
RUTA_CONTRATO = RAIZ / "contracts" / "data_contract.json"

DIR_RAW = RAIZ / "data" / "raw"
DIR_PROCESSED = RAIZ / "data" / "processed"
DIR_MODELOS = RAIZ / "models"
DIR_LOGS = RAIZ / "logs"
DIR_REPORTES = RAIZ / "reports"

RUTA_RAW = DIR_RAW / "iris.csv"
RUTA_TRAIN = DIR_PROCESSED / "train.csv"
RUTA_TEST = DIR_PROCESSED / "test.csv"
RUTA_MODELO = DIR_MODELOS / "model.pkl"
RUTA_BASELINE = DIR_MODELOS / "baseline.pkl"
RUTA_INFO_RUNS = DIR_MODELOS / "run_info.json"
RUTA_LOG = DIR_LOGS / "pipeline.log"
RUTA_REPORTE_VALIDACION = DIR_REPORTES / "validation_report.json"
RUTA_REPORTE_EVALUACION = DIR_REPORTES / "evaluation_report.json"


def cargar_json(ruta):
    with open(ruta, encoding="utf-8") as archivo:
        return json.load(archivo)


def guardar_json(datos, ruta):
    Path(ruta).parent.mkdir(parents=True, exist_ok=True)
    with open(ruta, "w", encoding="utf-8") as archivo:
        json.dump(datos, archivo, ensure_ascii=False, indent=2)


def cargar_params():
    return cargar_json(RUTA_PARAMS)


def cargar_contrato():
    return cargar_json(RUTA_CONTRATO)


def sha256_archivo(ruta):
    """Huella digital del archivo: si los datos cambian, el hash cambia."""
    h = hashlib.sha256()
    with open(ruta, "rb") as archivo:
        for bloque in iter(lambda: archivo.read(65536), b""):
            h.update(bloque)
    return h.hexdigest()


def versiones_entorno():
    """Versiones de Python y librerías que se registran como etiquetas en MLflow."""
    import mlflow
    import sklearn

    return {
        "python_version": platform.python_version(),
        "sklearn_version": sklearn.__version__,
        "mlflow_version": mlflow.__version__,
        "sistema_operativo": platform.system(),
    }


def configurar_log(nombre):
    """Logger que escribe en consola y en logs/pipeline.log con fecha y etapa."""
    DIR_LOGS.mkdir(parents=True, exist_ok=True)
    logger = logging.getLogger(nombre)
    if logger.handlers:
        return logger
    logger.setLevel(logging.INFO)
    formato = logging.Formatter("%(asctime)s | %(levelname)-7s | %(name)-9s | %(message)s",
                                datefmt="%Y-%m-%d %H:%M:%S")
    for manejador in (logging.StreamHandler(), logging.FileHandler(RUTA_LOG, encoding="utf-8")):
        manejador.setFormatter(formato)
        logger.addHandler(manejador)
    return logger


def preparar_entorno_mlflow():
    """Variables de entorno que silencian avisos informativos de MLflow y Git."""
    os.environ.setdefault("MLFLOW_DISABLE_AGENT_HINT", "1")
    os.environ.setdefault("GIT_PYTHON_REFRESH", "quiet")

Overwriting src/utils.py


## 5. Etapa 1 – Extracción y partición - `src/prepare.py`

En `src/prepare.py` se carga el conjunto de datos Iris desde scikit-learn, se cambian los nombres de las columnas para que coincidan con los definidos en el contrato y se guardan tanto los datos originales como su división en entrenamiento y prueba, utilizando la misma semilla de la Actividad 1.1.

En esta ocasión, la partición se realiza de forma **estratificada**, como se propuso en clase. Esto significa que se utilizan 40 flores de cada especie para entrenamiento y 10 de cada una para prueba. A diferencia de la Actividad 1.1, donde la distribución fue de 10, 9 y 11 flores en la prueba, ahora las tres especies tienen la misma representación. Esto permite hacer una evaluación más equilibrada, aunque los datos de prueba cambian y por lo tanto, los resultados son diferentes a los obtenidos anteriormente.

In [6]:
%%writefile src/prepare.py
"""Etapa 1 del pipeline: extracción y partición de los datos.

Carga el dataset Iris incluido en scikit-learn, renombra las columnas a los
nombres que define el contrato de datos y guarda:
  - data/raw/iris.csv              dataset completo (150 registros)
  - data/processed/train.csv       80 % para entrenamiento
  - data/processed/test.csv        20 % para prueba

La partición usa la misma semilla (42) y proporción (80/20) de la Actividad 1.1,
pero ahora es estratificada pues conserva en train y test la proporción de cada
especie (40/40/40 y 10/10/10), como se propuso en clase. En la Actividad
1.1 la partición no era estratificada y la prueba quedó con 10, 9 y 11 flores.
"""

import pandas as pd
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split

from src import utils

# Nombres originales de scikit-learn y nombres del contrato (sin espacios ni
# unidades, compatibles con la API de la Semana 3)

NOMBRES_COLUMNAS = {
    "sepal length (cm)": "sepal_length",
    "sepal width (cm)": "sepal_width",
    "petal length (cm)": "petal_length",
    "petal width (cm)": "petal_width",
}


def main():
    log = utils.configurar_log("prepare")
    params = utils.cargar_params()["datos"]

    iris = load_iris(as_frame=True)
    datos = iris.data.rename(columns=NOMBRES_COLUMNAS)
    datos["species"] = iris.target

    utils.DIR_RAW.mkdir(parents=True, exist_ok=True)
    utils.DIR_PROCESSED.mkdir(parents=True, exist_ok=True)
    datos.to_csv(utils.RUTA_RAW, index=False)
    log.info("Dataset crudo guardado: %s (%d registros, %d columnas)",
             utils.RUTA_RAW.relative_to(utils.RAIZ), *datos.shape)

    train, test = train_test_split(
        datos, test_size=params["test_size"], random_state=params["split_seed"],
        stratify=datos["species"] if params["estratificar"] else None,
    )
    train.to_csv(utils.RUTA_TRAIN, index=False)
    test.to_csv(utils.RUTA_TEST, index=False)

    conteo_test = test["species"].value_counts().sort_index().to_dict()
    log.info("Partición %d/%d con split_seed=%d (estratificada: %s) -> train: %d registros | "
             "test: %d registros", round((1 - params["test_size"]) * 100),
             round(params["test_size"] * 100), params["split_seed"],
             "sí" if params["estratificar"] else "no", len(train), len(test))
    log.info("Clases en test (0=setosa, 1=versicolor, 2=virginica): %s", conteo_test)
    log.info("SHA-256 train.csv: %s", utils.sha256_archivo(utils.RUTA_TRAIN))


if __name__ == "__main__":
    main()

Overwriting src/prepare.py


## 6. Etapa 2 – Validación de datos de entrada - `src/validate_data.py`

Tal como lo solicita la guía, se implementa **sin librerías externas**. Solo usa `csv`, `json`, `math` y `argparse`
de la librería estándar (no usa pandas, numpy ni Great Expectations).

Reglas que verifica en cada registro:

| Regla | Qué verifica |
|---|---|
| columnas | Están todas las columnas requeridas y no sobra ninguna |
| nulo | Ningún valor vacío: `None`, `""`, `NaN`, `"NA"`, `"null"`, etc. |
| tipo | Las medidas son números finitos. La especie es un entero |
| rango | Cada medida está entre su mínimo y máximo lógicos |
| dominio | La especie solo toma 0, 1 o 2 |
| relacion | Pétalo más corto que el sépalo |

Además aplica dos **reglas de conjunto**, que comparan los archivos entre sí:

| Regla | Qué verifica |
|---|---|
| clases_completas | `train.csv` y `test.csv` contienen las tres especies |
| sin_solapamiento_train_test | Ninguna flor de prueba está también en entrenamiento |

La segunda responde a la lección vista en clase, pues al copiar la prueba en el entrenamiento da un accuracy perfecto y pasa todas las pruebas, ya que un pipeline solo detecta lo que se le enseñó a revisar. Por ello, aquí se le enseña a revisarlo. Solo se tolera la flor que ya viene duplicada en el Iris original.

Decisiones de diseño:

- En un **CSV** todos los valores llegan como texto, por lo que se intenta convertirlos a número. En un
  **JSON** de inferencia el tipo ya viene definido. `"5.1"` entre comillas es texto y se rechaza.
- Los booleanos se rechazan siempre, dado que en Python `True` es una subclase de `int` y pasaría como 1.
- `inf` se trata como error de tipo, en cuanto no es una medida real.
- La regla entre campos solo se evalúa si ambos valores ya son válidos, con el fin de no reportar dos veces el mismo problema.
- Los duplicados se reportan como **advertencia**. Iris tiene una flor repetida y es un dato real.
- El script termina con código 1 si algo falla. Ese código es el que detiene el pipeline.

In [7]:
%%writefile src/validate_data.py
"""Etapa 2 del pipeline: validación de datos de entrada sin librerías externas.

Solo usa la librería estándar de Python (csv, json, math, argparse). Las reglas
no están escritas en el código, se leen de contracts/data_contract.json. De esta forma,
tanto el entrenamiento como la inferencia utilizan los mismos criterios para validar
los datos.

Se verifican en cada registro las reglas definidas en el contrato (columnas, nulo,
tipo, rango, dominio y relación).
A nivel de archivo se revisa que tenga registros y se reportan como advertencia
(no bloquean) los registros duplicados.
Además se tienen en cuenta las reglas de conjunto, que comparan los archivos entre sí.

Uso:
  python -m src.validate_data                         valida data/raw, train y test
  python -m src.validate_data --entrada archivo.json  valida una solicitud de inferencia
  python -m src.validate_data --ejemplos              valida los ejemplos del contrato
"""

import argparse
import csv
import json
import math
import sys
from datetime import datetime

from src import utils

# Textos que se interpretan como valor faltante al leer un CSV o un JSON
TEXTOS_NULOS = {"", "na", "n/a", "nan", "null", "none", "nulo", "-"}


def es_nulo(valor):
    if valor is None:
        return True
    if isinstance(valor, float) and math.isnan(valor):
        return True
    if isinstance(valor, str) and valor.strip().lower() in TEXTOS_NULOS:
        return True
    return False


def convertir_numero(valor, origen):
    """Devuelve el valor como float o None si no es un número válido.
    """

    if isinstance(valor, bool):
        return None
    if origen == "json":
        if isinstance(valor, (int, float)):
            numero = float(valor)
        else:
            return None
    else:
        try:
            numero = float(str(valor).strip())
        except ValueError:
            return None
    return numero if math.isfinite(numero) else None


def convertir_entero(valor, origen):
    numero = convertir_numero(valor, origen)
    if numero is None or not numero.is_integer():
        return None
    return int(numero)


def error(campo, regla, mensaje, valor):
    return {"campo": campo, "regla": regla, "mensaje": mensaje, "valor": repr(valor)}


def validar_registro(registro, contrato, incluir_objetivo=False, origen="json"):
    """Valida un registro (diccionario columna -> valor) y devuelve la lista de errores.

    incluir_objetivo=True se usa para los archivos de entrenamiento y prueba, que
    traen la columna species. En inferencia (False) esa columna está prohibida.
    """
    campos = contrato["entrada"]["campos"]
    objetivo = contrato["variable_objetivo"]
    esperadas = list(campos) + ([objetivo["nombre"]] if incluir_objetivo else [])
    errores = []

    for columna in esperadas:
        if columna not in registro:
            errores.append(error(columna, "columnas", "Falta la columna requerida", None))
    if not contrato["entrada"]["campos_adicionales_permitidos"]:
        for columna in registro:
            if columna not in esperadas:
                motivo = ("Campo no permitido en inferencia: es la variable objetivo (fuga de datos)"
                          if columna == objetivo["nombre"] else "Campo no definido en el contrato")
                errores.append(error(columna, "columnas", motivo, registro[columna]))

    numeros = {}
    for nombre, regla in campos.items():
        if nombre not in registro:
            continue
        valor = registro[nombre]
        if es_nulo(valor):
            if not regla["admite_nulos"]:
                errores.append(error(nombre, "nulo", "Valor nulo o vacío", valor))
            continue
        numero = convertir_numero(valor, origen)
        if numero is None:
            errores.append(error(nombre, "tipo", f"Se esperaba {regla['tipo']} y llegó "
                                 f"{type(valor).__name__}", valor))
            continue
        if not regla["minimo"] <= numero <= regla["maximo"]:
            errores.append(error(nombre, "rango", f"Fuera del rango lógico "
                                 f"[{regla['minimo']}, {regla['maximo']}] {regla['unidad']}", valor))
            continue
        numeros[nombre] = numero

    # Las reglas entre campos solo se evalúan si ambos valores ya pasaron las
    # validaciones individuales, para no reportar el mismo problema dos veces

    for relacion in contrato["entrada"]["reglas_entre_campos"]:
        menor, mayor = relacion["campo_menor"], relacion["campo_mayor"]
        if menor in numeros and mayor in numeros and not numeros[menor] < numeros[mayor]:
            errores.append(error(f"{menor}/{mayor}", "relacion", relacion["descripcion"],
                                 (numeros[menor], numeros[mayor])))

    if incluir_objetivo and objetivo["nombre"] in registro:
        valor = registro[objetivo["nombre"]]
        if es_nulo(valor):
            errores.append(error(objetivo["nombre"], "nulo", "Valor nulo o vacío", valor))
        else:
            clase = convertir_entero(valor, origen)
            if clase is None:
                errores.append(error(objetivo["nombre"], "tipo", "Se esperaba un entero", valor))
            elif clase not in objetivo["valores_permitidos"]:
                errores.append(error(objetivo["nombre"], "dominio",
                                     f"Clase fuera de {objetivo['valores_permitidos']}", valor))
    return errores


def validar_csv(ruta, contrato):
    """Valida un archivo CSV de entrenamiento o prueba.

    Devuelve el reporte del archivo y sus filas, que luego usan las reglas de conjunto.
    """
    reporte = {"archivo": str(ruta.relative_to(utils.RAIZ)).replace("\\", "/"),
               "sha256": utils.sha256_archivo(ruta), "registros": 0,
               "errores": [], "advertencias": []}
    with open(ruta, newline="", encoding="utf-8") as archivo:
        filas = list(csv.DictReader(archivo))
    vistos = {}
    # La fila 1 del archivo es el encabezado, por eso los datos empiezan en la 2
    for numero_fila, fila in enumerate(filas, start=2):
        reporte["registros"] += 1
        for e in validar_registro(fila, contrato, incluir_objetivo=True, origen="csv"):
            reporte["errores"].append({"fila": numero_fila, **e})
        clave = tuple(fila.values())
        if clave in vistos:
            reporte["advertencias"].append(
                f"Fila {numero_fila} duplica la fila {vistos[clave]} {list(fila.values())}")
        else:
            vistos[clave] = numero_fila
    if reporte["registros"] == 0:
        reporte["errores"].append({"fila": None, "campo": None, "regla": "vacio",
                                   "mensaje": "El archivo no contiene registros", "valor": None})
    reporte["aprobado"] = not reporte["errores"]
    return reporte, filas


def clave_flor(fila, contrato):
    """Representa una flor como tupla de números para compararla entre archivos.

    Devuelve None si algún valor no es numérico, esos casos ya los reporta la
    validación por registro.
    """
    columnas = list(contrato["entrada"]["campos"]) + [contrato["variable_objetivo"]["nombre"]]
    valores = [convertir_numero(fila.get(c), "csv") for c in columnas]
    return None if None in valores else tuple(valores)


def validar_conjuntos(raw, train, test, contrato):
    """Reglas que comparan los archivos entre sí y que una fila aislada no puede
    revelar."""

    reglas = {r["id"] for r in contrato["reglas_de_conjunto"]}
    objetivo = contrato["variable_objetivo"]
    errores = []

    if "clases_completas" in reglas:
        for nombre, filas in (("train.csv", train), ("test.csv", test)):
            presentes = {convertir_entero(f.get(objetivo["nombre"]), "csv") for f in filas}
            faltantes = [c for c in objetivo["valores_permitidos"] if c not in presentes]
            if faltantes:
                etiquetas = [objetivo["etiquetas"][str(c)] for c in faltantes]
                errores.append({"regla": "clases_completas", "archivo": nombre,
                                "mensaje": f"Faltan especies: {etiquetas}"})

    if "sin_solapamiento_train_test" in reglas:
        # Una flor que ya viene repetida en el dataset original puede quedar legítimamente
        # en ambos conjuntos. Cualquier otra coincidencia indica que se copiaron
        # registros

        conteo_raw = {}
        for f in raw:
            clave = clave_flor(f, contrato)
            conteo_raw[clave] = conteo_raw.get(clave, 0) + 1
        claves_train = {clave_flor(f, contrato) for f in train} - {None}
        compartidas = [clave_flor(f, contrato) for f in test
                       if clave_flor(f, contrato) in claves_train]
        no_explicadas = [c for c in compartidas if conteo_raw.get(c, 0) < 2]
        if no_explicadas:
            errores.append({"regla": "sin_solapamiento_train_test", "archivo": "train.csv/test.csv",
                            "mensaje": f"{len(no_explicadas)} de {len(test)} flores de prueba también "
                                       "están en entrenamiento (fuga de datos)"})
    return errores


def validar_solicitud(datos, contrato):
    """Valida una solicitud de inferencia: un objeto JSON o una lista de objetos."""
    registros = datos if isinstance(datos, list) else [datos]
    resultados = []
    for indice, registro in enumerate(registros):
        if not isinstance(registro, dict):
            errores = [error(None, "formato", "Cada registro debe ser un objeto JSON", registro)]
        else:
            errores = validar_registro(registro, contrato, incluir_objetivo=False, origen="json")
        resultados.append({"registro": indice, "aprobado": not errores, "errores": errores})
    return resultados


def resumir_errores(errores):
    """Cuenta los errores por regla, útil para leer el reporte de un vistazo."""
    conteo = {}
    for e in errores:
        conteo[e["regla"]] = conteo.get(e["regla"], 0) + 1
    return conteo


def validar_archivos_pipeline(log):
    contrato = utils.cargar_contrato()
    resultados = [validar_csv(ruta, contrato)
                  for ruta in (utils.RUTA_RAW, utils.RUTA_TRAIN, utils.RUTA_TEST)]
    reportes = [reporte for reporte, _ in resultados]
    for r in reportes:
        r["errores_por_regla"] = resumir_errores(r["errores"])
        estado = "APROBADO" if r["aprobado"] else "RECHAZADO"
        log.info("%s | %s | %d registros | %d errores %s | %d advertencias", estado,
                 r["archivo"], r["registros"], len(r["errores"]), r["errores_por_regla"] or "",
                 len(r["advertencias"]))
        for e in r["errores"][:10]:
            log.error("  fila %s | %s | %s | %s | valor=%s", e["fila"], e["campo"], e["regla"],
                      e["mensaje"], e["valor"])
        for a in r["advertencias"]:
            log.warning("  %s", a)

    errores_conjunto = validar_conjuntos(*(filas for _, filas in resultados), contrato)
    log.info("%s | reglas de conjunto: %s", "RECHAZADO" if errores_conjunto else "APROBADO",
             ", ".join(r["id"] for r in contrato["reglas_de_conjunto"]))
    for e in errores_conjunto:
        log.error("  %s | %s | %s", e["archivo"], e["regla"], e["mensaje"])

    aprobado = all(r["aprobado"] for r in reportes) and not errores_conjunto
    utils.guardar_json({
        "fecha": datetime.now().isoformat(timespec="seconds"),
        "contrato": f"{contrato['nombre']} v{contrato['version']}",
        "reglas": ["columnas", "nulo", "tipo", "rango", "dominio", "relacion"]
                  + [r["id"] for r in contrato["reglas_de_conjunto"]],
        "aprobado": aprobado,
        "archivos": reportes,
        "errores_de_conjunto": errores_conjunto,
    }, utils.RUTA_REPORTE_VALIDACION)
    log.info("Reporte guardado en %s", utils.RUTA_REPORTE_VALIDACION.relative_to(utils.RAIZ))
    return aprobado


def validar_ejemplos_contrato(log):
    """Comprueba que el contrato sea coherente. El ejemplo válido debe pasar y el
    inválido debe fallar exactamente con los errores que el contrato anuncia."""

    contrato = utils.cargar_contrato()
    valido = validar_solicitud(contrato["ejemplo_entrada_valida"], contrato)[0]
    invalido = validar_solicitud(contrato["ejemplo_entrada_invalida"], contrato)[0]
    obtenidos = {(e["campo"], e["regla"]) for e in invalido["errores"]}
    esperados = {(e["campo"], e["regla"]) for e in contrato["errores_esperados_entrada_invalida"]}

    log.info("Ejemplo válido   -> %s", "aprobado" if valido["aprobado"] else valido["errores"])
    log.info("Ejemplo inválido -> %d errores detectados", len(invalido["errores"]))
    for e in invalido["errores"]:
        log.info("  %s | %s | %s | valor=%s", e["campo"], e["regla"], e["mensaje"], e["valor"])
    coherente = valido["aprobado"] and not invalido["aprobado"] and obtenidos == esperados
    log.info("Contrato coherente con el validador: %s", "sí" if coherente else "no")
    return coherente


def main():
    parser = argparse.ArgumentParser(description="Validación de datos contra el contrato")
    grupo = parser.add_mutually_exclusive_group()
    grupo.add_argument("--entrada", help="Archivo JSON con una solicitud de inferencia")
    grupo.add_argument("--ejemplos", action="store_true",
                       help="Valida los ejemplos válido e inválido del contrato")
    args = parser.parse_args()
    log = utils.configurar_log("validate")

    if args.entrada:
        with open(args.entrada, encoding="utf-8") as archivo:
            resultados = validar_solicitud(json.load(archivo), utils.cargar_contrato())
        print(json.dumps(resultados, ensure_ascii=False, indent=2))
        correcto = all(r["aprobado"] for r in resultados)
    elif args.ejemplos:
        correcto = validar_ejemplos_contrato(log)
    else:
        correcto = validar_archivos_pipeline(log) and validar_ejemplos_contrato(log)

    # Código de salida 1 detiene el pipeline (Makefile y run_pipeline.py)
    sys.exit(0 if correcto else 1)

if __name__ == "__main__":
    main()

Overwriting src/validate_data.py


## 7. Etapa 3 – Entrenamiento con MLflow - `src/train.py`

En `src/train.py` se mantiene la lógica utilizada en la Actividad 1.1, incluyendo la conexión con SQLite, el experimento `Proyecto_Final_Grado`, las semillas, las etiquetas y `skops_trusted_types`. Sin embargo, se incorporan tres ajustes importantes:

1. **Control de calidad antes del entrenamiento.** Antes de iniciar, se comprueba que el reporte de validación esté aprobado y que el hash SHA-256 de `train.csv` coincida con el registrado durante la validación. Esto evita que el modelo se entrene con datos rechazados o modificados después de su revisión.

2. **Uso exclusivo de los datos de entrenamiento.** La validación cruzada se realiza únicamente con los 120 registros destinados al entrenamiento, sin utilizar los datos de prueba. Por esta razón, y por la partición estratificada, el resultado es diferente al de la Actividad 1.1, donde se utilizaron los 150 registros.

3. **Mayor seguimiento en MLflow.** Se guarda información adicional de cada ejecución, como el hash de los datos, la versión del contrato y la firma de entrada del modelo. También se incluyen como artefactos el contrato, el reporte de validación, la configuración y el archivo `requirements.txt`.

Finalmente, se entrenan tanto la baseline como el modelo seleccionado utilizando los mismos datos, lo que permite comparar sus resultados bajo las mismas condiciones.

In [8]:
%%writefile src/train.py
"""Etapa 3 del pipeline: entrenamiento y registro en MLflow.

Entrena dos modelos con los mismos datos y semillas:
  - Baseline (100 árboles, sin límite de profundidad): punto de referencia.
  - Modelo seleccionado en la Actividad 1.1 (120 árboles, profundidad 4).

Solo usa data/processed/train.csv. El conjunto de prueba se reserva para la
etapa de evaluación por lo que ninguna decisión de entrenamiento lo ve.

Antes de entrenar comprueba que el último reporte de validación esté aprobado y
que train.csv no haya cambiado desde que se validó (mismo SHA-256).
"""

import sys
import time

import joblib
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score
from sklearn.model_selection import cross_val_score

from src import utils

utils.preparar_entorno_mlflow()
import mlflow  # noqa: E402  (se importa después de configurar las variables de entorno)
import mlflow.sklearn  # noqa: E402
from mlflow.models import infer_signature  # noqa: E402


def verificar_validacion(log):
    """Bloquea el entrenamiento si los datos no pasaron la validación."""
    if not utils.RUTA_REPORTE_VALIDACION.exists():
        log.error("No existe %s. Ejecute primero la etapa validate.", utils.RUTA_REPORTE_VALIDACION.name)
        return False
    reporte = utils.cargar_json(utils.RUTA_REPORTE_VALIDACION)
    if not reporte["aprobado"]:
        log.error("La validación de datos fue rechazada. No se entrena el modelo.")
        return False
    hash_validado = {a["archivo"]: a["sha256"] for a in reporte["archivos"]}["data/processed/train.csv"]
    if hash_validado != utils.sha256_archivo(utils.RUTA_TRAIN):
        log.error("train.csv cambió después de la validación. Vuelva a ejecutar validate.")
        return False
    return True


def entrenar_y_registrar(config, X, y, params, contrato, etiquetas, log):
    semilla = params["entrenamiento"]["model_seed"]
    folds = params["entrenamiento"]["cv_folds"]

    with mlflow.start_run(run_name=config["run_name"]) as run:
        mlflow.log_params({
            "n_estimators": config["n_estimators"],
            "max_depth": config["max_depth"],
            "model_seed": semilla,
            "split_seed": params["datos"]["split_seed"],
            "test_size": params["datos"]["test_size"],
            "estratificado": params["datos"]["estratificar"],
            "cv_folds": folds,
            "train_registros": len(X),
            "train_sha256": utils.sha256_archivo(utils.RUTA_TRAIN),
            "contrato_version": contrato["version"],
        })
        mlflow.set_tags(etiquetas)

        def nuevo_modelo():
            return RandomForestClassifier(n_estimators=config["n_estimators"],
                                          max_depth=config["max_depth"], random_state=semilla)

        # La validación cruzada se hace solo sobre train para no usar el conjunto de prueba
        cv = cross_val_score(nuevo_modelo(), X, y, cv=folds)

        modelo = nuevo_modelo()
        inicio = time.time()
        modelo.fit(X, y)
        tiempo = time.time() - inicio

        pred_train = modelo.predict(X)
        metricas = {
            "train_accuracy": float((pred_train == y).mean()),
            "train_f1_macro": float(f1_score(y, pred_train, average="macro")),
            "cv_accuracy_mean": float(cv.mean()),
            "cv_accuracy_std": float(cv.std()),
            "train_time_s": tiempo,
        }
        mlflow.log_metrics(metricas)

        # La firma (signature) documenta en MLflow las mismas columnas y tipos del contrato
        firma = infer_signature(X, modelo.predict(X))
        mlflow.sklearn.log_model(modelo, name="iris_model", signature=firma,
                                 input_example=X.head(3),
                                 skops_trusted_types=["sklearn.tree._tree.Tree"])
        mlflow.log_artifact(str(utils.RUTA_CONTRATO), artifact_path="contrato")
        mlflow.log_artifact(str(utils.RUTA_REPORTE_VALIDACION), artifact_path="validacion")
        mlflow.log_artifact(str(utils.RUTA_PARAMS), artifact_path="config")
        mlflow.log_artifact(str(utils.RAIZ / "requirements.txt"))

        log.info("%-27s | train_acc=%.3f | cv=%.3f ± %.3f | run_id=%s", config["run_name"],
                 metricas["train_accuracy"], metricas["cv_accuracy_mean"],
                 metricas["cv_accuracy_std"], run.info.run_id)
        return modelo, run.info.run_id, metricas


def main():
    log = utils.configurar_log("train")
    if not verificar_validacion(log):
        sys.exit(1)

    params = utils.cargar_params()
    contrato = utils.cargar_contrato()
    columnas = contrato["modelo"]["orden_caracteristicas"]
    objetivo = contrato["variable_objetivo"]["nombre"]

    train = pd.read_csv(utils.RUTA_TRAIN)
    X, y = train[columnas], train[objetivo]

    mlflow.set_tracking_uri(params["mlflow"]["tracking_uri"])
    mlflow.set_experiment(params["mlflow"]["experimento"])
    etiquetas = {
        "autor": "Laura Tatiana Rincón López",
        "dataset": "Iris (sklearn.datasets.load_iris, 150 registros)",
        "modelo": "RandomForestClassifier",
        "etapa": "Seccion_2_pipeline",
        **utils.versiones_entorno(),
    }

    utils.DIR_MODELOS.mkdir(parents=True, exist_ok=True)
    info = {}
    for clave, ruta in (("baseline", utils.RUTA_BASELINE), ("modelo_seleccionado", utils.RUTA_MODELO)):
        modelo, run_id, metricas = entrenar_y_registrar(params[clave], X, y, params, contrato,
                                                        {**etiquetas, "rol": clave}, log)
        joblib.dump(modelo, ruta)
        info[clave] = {"run_name": params[clave]["run_name"], "run_id": run_id,
                       "artefacto": str(ruta.relative_to(utils.RAIZ)), "metricas_train": metricas}

    utils.guardar_json(info, utils.RUTA_INFO_RUNS)
    log.info("Modelos guardados en %s y run IDs en %s", utils.DIR_MODELOS.name,
             utils.RUTA_INFO_RUNS.name)


if __name__ == "__main__":
    main()

Overwriting src/train.py


## 8. Etapa 4 – Validación del modelo - `src/evaluate.py`

En `src/evaluate.py` se evalúan los dos modelos utilizando las 30 flores de prueba y se comparan sus resultados con los valores mínimos definidos en `config/params.json`.

Para aprobar el modelo seleccionado se deben cumplir las siguientes condiciones:

- Obtener un accuracy, F1 macro y accuracy medio de validación cruzada iguales o superiores a 0.90.
- Tener un desempeño igual o mejor que la baseline.
- Generar una predicción para el ejemplo válido del contrato, respetando el formato de salida esperado.
- Rechazar el ejemplo inválido antes de que llegue al modelo.

Los resultados de esta evaluación se guardan en los mismos runs de MLflow creados durante el entrenamiento. Además, el modelo seleccionado recibe la etiqueta `validacion_modelo`, que indica si fue `aprobada` o `rechazada` según el cumplimiento de los criterios establecidos.

In [9]:
%%writefile src/evaluate.py
"""Etapa 4 del pipeline: validación del modelo sobre el conjunto de prueba.

Esta etapa se encarga de evaluar la baseline y el modelo seleccionado utilizando las
mismas 30 flores de prueba y comparando sus resultados con los mínimos de calidad
definidos en config/params.json. Si el modelo seleccionado no cumple alguno de estos
requisitos, el proceso se detiene y genera un error.

También se verifica que el modelo cumpla con las reglas del contrato de datos. Para
ello, se comprueba que el ejemplo válido genere una predicción con el formato esperado
y que el ejemplo inválido sea rechazado antes de llegar al modelo.

Por último, las métricas obtenidas en las pruebas se agregan a los mismos registros de
MLflow que se crearon durante el entrenamiento, permitiendo consultar todo el proceso
en un solo lugar.
"""

import sys

import joblib
import pandas as pd
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score

from src import utils
from src.predict import predecir

utils.preparar_entorno_mlflow()
import mlflow  # noqa: E402


def evaluar(modelo, X, y):
    pred = modelo.predict(X)
    return {
        "test_accuracy": float(accuracy_score(y, pred)),
        "test_f1_macro": float(f1_score(y, pred, average="macro")),
        "test_errores": int((pred != y).sum()),
        "matriz_confusion": confusion_matrix(y, pred, labels=[0, 1, 2]).tolist(),
    }


def main():
    log = utils.configurar_log("evaluate")
    params = utils.cargar_params()
    contrato = utils.cargar_contrato()
    info = utils.cargar_json(utils.RUTA_INFO_RUNS)
    umbrales = params["umbrales_calidad"]

    test = pd.read_csv(utils.RUTA_TEST)
    X = test[contrato["modelo"]["orden_caracteristicas"]]
    y = test[contrato["variable_objetivo"]["nombre"]]

    mlflow.set_tracking_uri(params["mlflow"]["tracking_uri"])
    resultados = {}
    for clave, ruta in (("baseline", utils.RUTA_BASELINE), ("modelo_seleccionado", utils.RUTA_MODELO)):
        metricas = evaluar(joblib.load(ruta), X, y)
        metricas["cv_accuracy_mean"] = info[clave]["metricas_train"]["cv_accuracy_mean"]
        metricas["train_accuracy"] = info[clave]["metricas_train"]["train_accuracy"]
        metricas["brecha_train_test"] = metricas["train_accuracy"] - metricas["test_accuracy"]
        resultados[clave] = metricas
        with mlflow.start_run(run_id=info[clave]["run_id"]):
            mlflow.log_metrics({k: metricas[k] for k in
                                ("test_accuracy", "test_f1_macro", "test_errores", "brecha_train_test")})
            if clave == "baseline":
                mlflow.set_tag("validacion_modelo", "referencia")
        log.info("%-27s | test_acc=%.3f | f1_macro=%.3f | errores=%d/%d | brecha=%.3f",
                 info[clave]["run_name"], metricas["test_accuracy"], metricas["test_f1_macro"],
                 metricas["test_errores"], len(y), metricas["brecha_train_test"])

    seleccionado = resultados["modelo_seleccionado"]
    controles = {
        "accuracy_min": seleccionado["test_accuracy"] >= umbrales["accuracy_min"],
        "f1_macro_min": seleccionado["test_f1_macro"] >= umbrales["f1_macro_min"],
        "cv_accuracy_min": seleccionado["cv_accuracy_mean"] >= umbrales["cv_accuracy_min"],
        "no_inferior_al_baseline": (seleccionado["test_accuracy"]
                                    >= resultados["baseline"]["test_accuracy"]),
    }

    salida_valida = predecir(contrato["ejemplo_entrada_valida"])[0]
    salida_invalida = predecir(contrato["ejemplo_entrada_invalida"])[0]
    controles["contrato_ejemplo_valido_predicho"] = (
        salida_valida.get("prediction") in contrato["salida"]["campos"]["prediction"]["valores_posibles"])
    controles["contrato_ejemplo_invalido_rechazado"] = "errores" in salida_invalida
    log.info("Ejemplo válido del contrato   -> %s", salida_valida)
    log.info("Ejemplo inválido del contrato -> rechazado con %d errores",
             len(salida_invalida.get("errores", [])))

    for control, cumple in controles.items():
        (log.info if cumple else log.error)("Control %-36s %s", control, "OK" if cumple else "FALLA")
    aprobado = all(controles.values())

    utils.guardar_json({"umbrales": umbrales, "resultados": resultados, "controles": controles,
                        "ejemplo_valido_salida": salida_valida, "aprobado": aprobado},
                       utils.RUTA_REPORTE_EVALUACION)
    with mlflow.start_run(run_id=info["modelo_seleccionado"]["run_id"]):
        mlflow.log_artifact(str(utils.RUTA_REPORTE_EVALUACION), artifact_path="evaluacion")
        mlflow.set_tag("validacion_modelo", "aprobada" if aprobado else "rechazada")

    log.info("Validación del modelo: %s", "APROBADA" if aprobado else "RECHAZADA")
    sys.exit(0 if aprobado else 1)


if __name__ == "__main__":
    main()

Overwriting src/evaluate.py


## 9. Inferencia con contrato - `src/predict.py`

En `src/predict.py` se validan los datos antes de realizar cualquier predicción, utilizando las mismas reglas definidas en el contrato. Si algún registro no cumple con ellas, se muestran los errores correspondientes y no se envía al modelo.

Esto permite asegurar que el modelo siempre reciba datos con las mismas columnas, tipos y rangos utilizados durante su entrenamiento, evitando diferencias entre los datos del entrenamiento y los que recibe al momento de utilizarlo (*training-serving skew*).

La respuesta incluye los campos `prediction` y `class_id`, siguiendo el mismo formato de la API de la Semana 3 del portal MLOps Santoto.

In [10]:
%%writefile src/predict.py
"""Inferencia con el modelo entrenado aplicando el contrato de datos.

Cada registro se valida contra el contrato antes de llegar al modelo. Si incumple
alguna regla no se predice y se devuelven los errores. Así el modelo solo recibe
datos con las mismas columnas, tipos y rangos con los que fue entrenado, lo que
evita el sesgo entre entrenamiento y servicio (training-serving skew).

Uso:
  python -m src.predict --ejemplo valida
  python -m src.predict --ejemplo invalida
  python -m src.predict --archivo solicitud.json
"""

import argparse
import json

import joblib
import pandas as pd

from src import utils
from src.validate_data import validar_solicitud


def predecir(datos):
    """Recibe un objeto o una lista de objetos JSON y devuelve una salida por registro."""
    contrato = utils.cargar_contrato()
    columnas = contrato["modelo"]["orden_caracteristicas"]
    etiquetas = contrato["variable_objetivo"]["etiquetas"]
    registros = datos if isinstance(datos, list) else [datos]
    modelo = joblib.load(utils.RUTA_MODELO)

    salidas = []
    for registro, validacion in zip(registros, validar_solicitud(registros, contrato)):
        if not validacion["aprobado"]:
            salidas.append({"errores": validacion["errores"]})
            continue
        fila = pd.DataFrame([registro], columns=columnas).astype(float)
        clase = int(modelo.predict(fila)[0])
        salidas.append({"prediction": etiquetas[str(clase)], "class_id": clase})
    return salidas


def main():
    parser = argparse.ArgumentParser(description="Predicción de especie Iris según el contrato")
    grupo = parser.add_mutually_exclusive_group(required=True)
    grupo.add_argument("--ejemplo", choices=["valida", "invalida"],
                       help="Usa el ejemplo de entrada válida o inválida del contrato")
    grupo.add_argument("--archivo", help="Archivo JSON con uno o varios registros")
    args = parser.parse_args()

    if args.ejemplo:
        datos = utils.cargar_contrato()[f"ejemplo_entrada_{args.ejemplo}"]
    else:
        datos = utils.cargar_json(args.archivo)
    print(json.dumps(predecir(datos), ensure_ascii=False, indent=2))


if __name__ == "__main__":
    main()

Overwriting src/predict.py


## 10. Automatización - `run_pipeline.py` y `Makefile`

En este punto se busca que todo el proceso pueda ejecutarse con un solo comando.

- `python run_pipeline.py` Permite ejecutar todas las etapas en cualquier sistema operativo. Cada una se ejecuta por separado, se registra cuánto tarda y, si ocurre algún error, el proceso se detiene sin continuar con las siguientes etapas, siguiendo el principio de fallar rápido. La opción `--reiniciar` elimina los resultados anteriores y vuelve a ejecutar todo desde cero, mientras que `--sin-preparar` permite omitir la preparación de los datos y trabajar con los archivos existentes, opción utilizada en el juego de la sección 17.

- `make pipeline` Permite ejecutar el mismo proceso mediante un Makefile, donde cada etapa solo se ejecuta si la anterior finalizó correctamente (`prepare → validate → train → evaluate → test`). Como Windows no incluye `make` por defecto, se dejan disponibles las dos alternativas.

In [11]:
%%writefile run_pipeline.py
"""Orquestador del pipeline: ejecuta todas las etapas con un solo comando.

    python run_pipeline.py              ejecuta el pipeline completo
    python run_pipeline.py --reiniciar  borra los resultados previos y reproduce desde cero
    python run_pipeline.py --sin-preparar  omite prepare y valida los datos tal como están
                                           (se usa con romper_datos.py)

Etapas, en orden:
  1. prepare   extracción y partición 80/20 de los datos
  2. validate  validación de datos contra el contrato (tipos, nulos, rangos)
  3. train     entrenamiento de la baseline y del modelo seleccionado con registro en MLflow
  4. evaluate  validación del modelo con umbrales de calidad sobre el conjunto de prueba
  5. test      pruebas automatizadas con pytest

Cada etapa corre en un proceso independiente. Si una termina con error, el
pipeline se detiene y las etapas siguientes no se ejecutan, de modo que nunca se
entrena con datos rechazados ni se da por válido un modelo que no pasó los umbrales.

Es el equivalente en Python de `make pipeline`, útil en Windows, donde make no
viene instalado por defecto.
"""

import argparse
import shutil
import subprocess
import sys
import time

from src import utils

ETAPAS = [
    ("prepare", [sys.executable, "-m", "src.prepare"]),
    ("validate", [sys.executable, "-m", "src.validate_data"]),
    ("train", [sys.executable, "-m", "src.train"]),
    ("evaluate", [sys.executable, "-m", "src.evaluate"]),
    ("test", [sys.executable, "-m", "pytest", "-q", "tests"]),
]

# Resultados generados por el pipeline; se eliminan con --reiniciar
GENERADOS = ["mlflow.db", "mlruns", "data", "models", "reports", "logs"]


def reiniciar():
    for nombre in GENERADOS:
        ruta = utils.RAIZ / nombre
        if ruta.is_dir():
            shutil.rmtree(ruta)
        elif ruta.exists():
            ruta.unlink()


def main():
    parser = argparse.ArgumentParser(description="Pipeline único de clasificación de Iris")
    parser.add_argument("--reiniciar", action="store_true",
                        help="Elimina mlflow.db, mlruns, data, models, reports y logs antes de ejecutar")
    parser.add_argument("--sin-preparar", action="store_true",
                        help="Omite la etapa prepare para validar los datos actuales sin regenerarlos")
    args = parser.parse_args()
    etapas = [e for e in ETAPAS if not (args.sin_preparar and e[0] == "prepare")]

    if args.reiniciar:
        reiniciar()
    log = utils.configurar_log("pipeline")
    if args.reiniciar:
        log.info("Resultados previos eliminados: %s", ", ".join(GENERADOS))

    log.info("Inicio del pipeline (%d etapas)", len(etapas))
    inicio_total = time.time()
    for numero, (nombre, comando) in enumerate(etapas, start=1):
        log.info("Etapa %d/%d: %s", numero, len(etapas), nombre)
        inicio = time.time()
        resultado = subprocess.run(comando, cwd=utils.RAIZ)
        duracion = time.time() - inicio
        if resultado.returncode != 0:
            log.error("Etapa %s falló (código %d) tras %.1f s. Pipeline detenido.",
                      nombre, resultado.returncode, duracion)
            sys.exit(resultado.returncode)
        log.info("Etapa %s completada en %.1f s", nombre, duracion)

    log.info("Pipeline completado en %.1f s. Consulte los runs con: "
             "mlflow ui --backend-store-uri sqlite:///mlflow.db", time.time() - inicio_total)


if __name__ == "__main__":
    main()

Overwriting run_pipeline.py


In [12]:
%%writefile Makefile
# Automatización del pipeline. Uso: make <objetivo>
# En Windows: python run_pipeline.py hace lo mismo que make pipeline.

PYTHON ?= python

.PHONY: help install pipeline prepare validate train evaluate test predict ui clean reproducir romper

help:
	@echo "make install     instala las dependencias de requirements.txt"
	@echo "make pipeline    prepare -> validate -> train -> evaluate -> test"
	@echo "make reproducir  borra resultados previos y ejecuta el pipeline desde cero"
	@echo "make prepare     extrae el dataset y crea la partición 80/20"
	@echo "make validate    valida los datos contra contracts/data_contract.json"
	@echo "make train       entrena la baseline y el modelo seleccionado (MLflow)"
	@echo "make evaluate    valida el modelo con los umbrales de calidad"
	@echo "make test        ejecuta las pruebas con pytest"
	@echo "make predict     predice el ejemplo válido e inválido del contrato"
	@echo "make romper E=3  aplica el sabotaje 3 (1 a 7) y ejecuta el pipeline sin prepare"
	@echo "make ui          abre MLflow UI en http://127.0.0.1:5000"
	@echo "make clean       elimina los resultados generados"

install:
	$(PYTHON) -m pip install -r requirements.txt

# Cada objetivo depende del anterior: si una etapa falla, make se detiene
pipeline: test

prepare:
	$(PYTHON) -m src.prepare

validate: prepare
	$(PYTHON) -m src.validate_data

train: validate
	$(PYTHON) -m src.train

evaluate: train
	$(PYTHON) -m src.evaluate

test: evaluate
	$(PYTHON) -m pytest -q tests

reproducir:
	$(PYTHON) run_pipeline.py --reiniciar

predict:
	$(PYTHON) -m src.predict --ejemplo valida
	$(PYTHON) -m src.predict --ejemplo invalida

# Juego de la Sesión 4: el pipeline debe detenerse en la etapa que detecta el sabotaje
romper:
	$(PYTHON) romper_datos.py $(E)
	$(PYTHON) run_pipeline.py --sin-preparar

ui:
	$(PYTHON) -m mlflow ui --backend-store-uri sqlite:///mlflow.db --host 127.0.0.1 --port 5000

clean:
	$(PYTHON) -c "import run_pipeline; run_pipeline.reiniciar()"

Overwriting Makefile


El archivo `romper_datos.py` permite recrear los siete escenarios de sabotaje del juego «¡Rompe el pipeline!» visto en clase. Para ello, primero genera nuevamente los datos originales y luego modifica `train.csv` según el escenario que se quiera probar. Su uso se explica en la sección 17.

In [13]:
%%writefile romper_datos.py
"""Sabotajes del juego "¡Rompe el pipeline!" visto en clase.

Regenera datos limpios con la etapa prepare y luego daña data/processed/train.csv
según el escenario elegido. Después se ejecuta el pipeline sin la etapa prepare
para ver qué control lo detiene:

    python romper_datos.py 3
    python run_pipeline.py --sin-preparar

Para volver a los datos correctos basta con ejecutar el pipeline completo
(python run_pipeline.py), porque prepare regenera los archivos.

Escenarios y control que los detiene en este repositorio:
  1  "cinco" en lugar de un número               validate_data · tipo
  2  celda vacía en petal_width                  validate_data · nulo
  3  un registro digitado en milímetros          validate_data · rango (4 errores)
  4  especie con código 3                        validate_data · dominio
  5  la extracción no trajo flores virginica     validate_data · clases_completas
  6  40 % de las etiquetas mal asignadas         evaluate · umbrales de accuracy
  7  las 30 flores de prueba copiadas en train   validate_data · sin_solapamiento_train_test
"""

import argparse

import numpy as np
import pandas as pd

from src import prepare, utils

ESCENARIOS = {
    1: "«cinco» en lugar de un número",
    2: "Celda vacía en petal_width",
    3: "Un registro digitado en milímetros",
    4: "Especie con código 3",
    5: "La extracción no trajo flores virginica",
    6: "40 % de las etiquetas mal asignadas",
    7: "Las 30 flores de prueba copiadas en entrenamiento",
}


def sabotear(escenario, train, test):
    medidas = ["sepal_length", "sepal_width", "petal_length", "petal_width"]
    train = train.astype(object)
    if escenario == 1:
        train.loc[0, "sepal_length"] = "cinco"
    elif escenario == 2:
        train.loc[0, "petal_width"] = ""
    elif escenario == 3:
        # Primera flor con pétalo de al menos 1 cm, para que las cuatro medidas en mm superen el máximo
        fila = train.index[train["petal_width"].astype(float) >= 1.0][0]
        train.loc[fila, medidas] = train.loc[fila, medidas].astype(float) * 10
    elif escenario == 4:
        train.loc[0, "species"] = 3
    elif escenario == 5:
        train = train[train["species"] != 2]
    elif escenario == 6:
        # Se cambia la etiqueta del 40 % de las flores por otra especie distinta
        rng = np.random.default_rng(0)
        indices = rng.choice(len(train), size=int(0.4 * len(train)), replace=False)
        for i in indices:
            actual = int(train.iloc[i]["species"])
            train.iloc[i, train.columns.get_loc("species")] = rng.choice(
                [c for c in (0, 1, 2) if c != actual])
    elif escenario == 7:
        train = pd.concat([train, test], ignore_index=True)
    return train


def main():
    parser = argparse.ArgumentParser(description="Daña train.csv según un escenario del juego")
    parser.add_argument("escenario", type=int, choices=sorted(ESCENARIOS))
    args = parser.parse_args()

    prepare.main()
    train = pd.read_csv(utils.RUTA_TRAIN)
    test = pd.read_csv(utils.RUTA_TEST)
    sabotear(args.escenario, train, test).to_csv(utils.RUTA_TRAIN, index=False)

    log = utils.configurar_log("romper")
    log.warning("Escenario %d aplicado a train.csv: %s", args.escenario, ESCENARIOS[args.escenario])
    log.warning("Ejecute: python run_pipeline.py --sin-preparar")


if __name__ == "__main__":
    main()

Overwriting romper_datos.py


## 11. Pruebas automatizadas, dependencias y archivos ignorados

- `tests/test_validate.py` Contiene las pruebas del validador. Cada prueba genera un registro o un grupo de archivos con algún error específico y comprueba que se identifique con la regla correcta.
- `tests/test_model.py` Reúne las pruebas para verificar que el pipeline funcione como se espera, revisando la partición de los datos, las columnas, el modelo, el cumplimiento del accuracy mínimo y el formato de las predicciones.
- `requirements.txt` Contiene las versiones fijas de las librerías utilizadas para desarrollar el proyecto y obtener los resultados.
- `.gitignore` Evita que se suban a GitHub archivos de datos, modelos, reportes, logs y la base de datos de MLflow, ya que estos se generan nuevamente al ejecutar el pipeline.

Las pruebas son la última etapa del pipeline, siguiendo la misma lógica del archivo `dvc.yaml` del portal.

In [14]:
%%writefile tests/test_validate.py
"""Pruebas unitarias del validador de datos (src/validate_data.py).

Cada prueba construye un registro o un grupo de archivos con un único problema y
verifica que el validador lo detecte con la regla correcta. No dependen de que el
pipeline se haya ejecutado antes.
"""

import pytest

from src import utils
from src.validate_data import es_nulo, validar_conjuntos, validar_registro, validar_solicitud

CONTRATO = utils.cargar_contrato()
VALIDO = CONTRATO["ejemplo_entrada_valida"]


def reglas(errores):
    return {(e["campo"], e["regla"]) for e in errores}


def test_ejemplo_valido_del_contrato_pasa():
    assert validar_registro(VALIDO, CONTRATO) == []


def test_ejemplo_invalido_del_contrato_falla_con_los_errores_esperados():
    errores = validar_registro(CONTRATO["ejemplo_entrada_invalida"], CONTRATO)
    esperados = {(e["campo"], e["regla"]) for e in CONTRATO["errores_esperados_entrada_invalida"]}
    assert reglas(errores) == esperados


@pytest.mark.parametrize("valor", [None, "", "  ", "NA", "nan", "null", float("nan")])
def test_detecta_nulos(valor):
    assert es_nulo(valor)
    assert ("sepal_width", "nulo") in reglas(validar_registro({**VALIDO, "sepal_width": valor}, CONTRATO))


@pytest.mark.parametrize("valor", ["5.1", "cinco", True, [5.1], {"cm": 5.1}])
def test_rechaza_tipos_no_numericos_en_json(valor):
    errores = validar_registro({**VALIDO, "sepal_length": valor}, CONTRATO)
    assert ("sepal_length", "tipo") in reglas(errores)


def test_acepta_texto_numerico_en_csv():
    fila = {k: str(v) for k, v in VALIDO.items()}
    assert validar_registro(fila, CONTRATO, origen="csv") == []


@pytest.mark.parametrize("campo,valor", [("sepal_length", 3.9), ("sepal_length", 8.1),
                                         ("petal_width", 0.0), ("petal_length", -1.0),
                                         ("sepal_width", float("inf"))])
def test_detecta_valores_fuera_de_rango_o_infinitos(campo, valor):
    errores = validar_registro({**VALIDO, campo: valor}, CONTRATO)
    assert reglas(errores) & {(campo, "rango"), (campo, "tipo")}


def test_detecta_columna_faltante_y_columna_extra():
    registro = {k: v for k, v in VALIDO.items() if k != "petal_width"}
    registro["color"] = "morado"
    assert reglas(validar_registro(registro, CONTRATO)) == {("petal_width", "columnas"),
                                                            ("color", "columnas")}


def test_rechaza_variable_objetivo_en_inferencia():
    errores = validar_registro({**VALIDO, "species": 1}, CONTRATO)
    assert ("species", "columnas") in reglas(errores)


@pytest.mark.parametrize("valor,regla", [("3", "dominio"), ("1.5", "tipo"), ("", "nulo")])
def test_valida_variable_objetivo_en_entrenamiento(valor, regla):
    fila = {**{k: str(v) for k, v in VALIDO.items()}, "species": valor}
    errores = validar_registro(fila, CONTRATO, incluir_objetivo=True, origen="csv")
    assert ("species", regla) in reglas(errores)


def test_detecta_columnas_intercambiadas_con_la_regla_entre_campos():
    registro = {**VALIDO, "sepal_length": 4.5, "petal_length": 6.0}
    assert ("petal_length/sepal_length", "relacion") in reglas(validar_registro(registro, CONTRATO))


def test_solicitud_por_lotes_valida_cada_registro():
    resultados = validar_solicitud([VALIDO, CONTRATO["ejemplo_entrada_invalida"], "texto"], CONTRATO)
    assert [r["aprobado"] for r in resultados] == [True, False, False]


def flor(sl, sw, pl, pw, especie):
    return {"sepal_length": str(sl), "sepal_width": str(sw), "petal_length": str(pl),
            "petal_width": str(pw), "species": str(especie)}


SETOSA, VERSICOLOR, VIRGINICA = flor(5.1, 3.5, 1.4, 0.2, 0), flor(6.0, 2.9, 4.5, 1.4, 1), flor(6.5, 3.0, 5.5, 2.0, 2)
OTRA_SETOSA, OTRA_VERSICOLOR, OTRA_VIRGINICA = (flor(4.9, 3.0, 1.4, 0.2, 0), flor(5.7, 2.8, 4.1, 1.3, 1),
                                                flor(6.3, 3.3, 6.0, 2.5, 2))


def test_conjuntos_correctos_no_generan_errores():
    train, test = [SETOSA, VERSICOLOR, VIRGINICA], [OTRA_SETOSA, OTRA_VERSICOLOR, OTRA_VIRGINICA]
    assert validar_conjuntos(train + test, train, test, CONTRATO) == []


def test_detecta_especie_ausente_en_entrenamiento():
    train, test = [SETOSA, VERSICOLOR], [OTRA_SETOSA, OTRA_VERSICOLOR, OTRA_VIRGINICA]
    errores = validar_conjuntos(train + test + [VIRGINICA], train, test, CONTRATO)
    assert [e["regla"] for e in errores] == ["clases_completas"]


def test_detecta_flores_de_prueba_copiadas_en_entrenamiento():
    test = [OTRA_SETOSA, OTRA_VERSICOLOR, OTRA_VIRGINICA]
    train = [SETOSA, VERSICOLOR, VIRGINICA] + test
    errores = validar_conjuntos(train, train, test, CONTRATO)
    assert [e["regla"] for e in errores] == ["sin_solapamiento_train_test"]


def test_tolera_una_flor_duplicada_en_el_dataset_original():
    train, test = [SETOSA, VERSICOLOR, VIRGINICA], [SETOSA, OTRA_VERSICOLOR, OTRA_VIRGINICA]
    raw = [SETOSA, SETOSA, VERSICOLOR, VIRGINICA, OTRA_VERSICOLOR, OTRA_VIRGINICA]
    assert validar_conjuntos(raw, train, test, CONTRATO) == []

Overwriting tests/test_validate.py


In [15]:
%%writefile tests/test_model.py
"""Pruebas sobre los resultados del pipeline (datos procesados y modelo).

Se ejecutan como última etapa del pipeline, después de prepare, validate, train
y evaluate, dado que revisan los archivos que esas etapas generan.
"""

import joblib
import pandas as pd

from src import utils
from src.predict import predecir

CONTRATO = utils.cargar_contrato()
COLUMNAS = CONTRATO["modelo"]["orden_caracteristicas"]


def test_particion_reproducible():
    train, test = pd.read_csv(utils.RUTA_TRAIN), pd.read_csv(utils.RUTA_TEST)
    assert (len(train), len(test)) == (120, 30)
    # La partición estratificada conserva la proporción de especies: 40/40/40 y 10/10/10
    assert train["species"].value_counts().sort_index().tolist() == [40, 40, 40]
    assert test["species"].value_counts().sort_index().tolist() == [10, 10, 10]


def test_columnas_coinciden_con_el_contrato():
    assert list(pd.read_csv(utils.RUTA_TRAIN).columns) == COLUMNAS + ["species"]


def test_validacion_de_datos_aprobada():
    assert utils.cargar_json(utils.RUTA_REPORTE_VALIDACION)["aprobado"]


def test_modelo_con_hiperparametros_del_modelo_seleccionado():
    modelo = joblib.load(utils.RUTA_MODELO)
    config = utils.cargar_params()["modelo_seleccionado"]
    assert (modelo.n_estimators, modelo.max_depth) == (config["n_estimators"], config["max_depth"])
    assert list(modelo.feature_names_in_) == COLUMNAS


def test_accuracy_supera_el_umbral():
    modelo = joblib.load(utils.RUTA_MODELO)
    test = pd.read_csv(utils.RUTA_TEST)
    umbral = utils.cargar_params()["umbrales_calidad"]["accuracy_min"]
    assert modelo.score(test[COLUMNAS], test["species"]) >= umbral


def test_salida_de_inferencia_cumple_el_contrato():
    salida = predecir(CONTRATO["ejemplo_entrada_valida"])[0]
    assert set(salida) == set(CONTRATO["salida"]["campos"])
    assert salida["prediction"] == CONTRATO["variable_objetivo"]["etiquetas"][str(salida["class_id"])]


def test_inferencia_rechaza_la_entrada_invalida():
    assert "errores" in predecir(CONTRATO["ejemplo_entrada_invalida"])[0]

Overwriting tests/test_model.py


In [16]:
%%writefile pytest.ini
[pytest]
pythonpath = .
testpaths = tests

Overwriting pytest.ini


In [17]:
%%writefile requirements.txt
# Versiones fijas para reproducir exactamente el entorno del experimento (Python 3.12)
mlflow==3.18.0
scikit-learn==1.9.1
pandas==3.0.5
joblib==1.6.0
pytest==9.1.1
ipykernel

Overwriting requirements.txt


In [18]:
%%writefile .gitignore
# Resultados que el pipeline regenera con: python run_pipeline.py --reiniciar
data/
models/
reports/
logs/
mlruns/
mlflow.db

# Entorno y cachés de Python
.venv/
venv/
__pycache__/
*.pyc
.pytest_cache/
.ipynb_checkpoints/

Overwriting .gitignore


# Parte 2. Ejecución y evidencias

## 12. Ejecución del pipeline completo con un solo comando

La función `ejecutar` permite correr un comando utilizando el mismo intérprete de Python del notebook y mostrar el resultado en el orden en que ocurre.

La opción `--reiniciar` elimina los archivos y carpetas generados anteriormente (`mlflow.db`, `mlruns`, `data`, `models`, `reports` y `logs`), para que el proceso comience desde cero y se pueda comprobar el resultado que obtendría cualquier persona al clonar el repositorio.

En Windows, no es posible eliminar `mlflow.db` mientras otro proceso lo esté utilizando. Por eso, si MLflow UI está abierto (sección 19), es necesario cerrarlo antes de volver a ejecutar esta celda. Si ya se ejecutó esa sección durante la misma sesión, también es recomendable reiniciar el kernel.

In [19]:
def ejecutar(*argumentos):
    # Ejecuta con el intérprete del notebook y en modo UTF-8 para que las tildes se
    # vean igual en Windows.
    # stderr=subprocess.STDOUT une el log y la salida de pytest en un solo flujo, en
    # orden cronológico.
    
    entorno = {**os.environ, "PYTHONUTF8": "1"}
    resultado = subprocess.run([sys.executable, *argumentos], cwd=RAIZ, env=entorno,
                               stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                               text=True, encoding="utf-8")
    print(resultado.stdout)
    print("Código de salida:", resultado.returncode)

ejecutar("run_pipeline.py", "--reiniciar")

2026-11-24 17:45:34 | INFO    | pipeline  | Resultados previos eliminados: mlflow.db, mlruns, data, models, reports, logs
2026-11-24 17:45:34 | INFO    | pipeline  | Inicio del pipeline (5 etapas)
2026-11-24 17:45:34 | INFO    | pipeline  | Etapa 1/5: prepare
2026-11-24 17:45:40 | INFO    | prepare   | Dataset crudo guardado: data\raw\iris.csv (150 registros, 5 columnas)
2026-11-24 17:45:40 | INFO    | prepare   | Partición 80/20 con split_seed=42 (estratificada: sí) -> train: 120 registros | test: 30 registros
2026-11-24 17:45:40 | INFO    | prepare   | Clases en test (0=setosa, 1=versicolor, 2=virginica): {0: 10, 1: 10, 2: 10}
2026-11-24 17:45:40 | INFO    | prepare   | SHA-256 train.csv: 99cf305e7ab13a559a3e15802ddc416be934d4f76af3b40d6d66be81f22ff517
2026-11-24 17:45:41 | INFO    | pipeline  | Etapa prepare completada en 6.9 s
2026-11-24 17:45:41 | INFO    | pipeline  | Etapa 2/5: validate
2026-11-24 17:45:41 | INFO    | validate  | APROBADO | data/raw/iris.csv | 150 registros | 0 

Las cinco etapas se ejecutan correctamente y finalizan sin errores (código 0). En el log queda registrado todo el proceso, desde la división de los datos y la validación de los tres archivos, hasta el entrenamiento con sus respectivos Run ID, las verificaciones de calidad del modelo y las pruebas finales.

## 13. Reporte de validación de datos

In [20]:
reporte_validacion = json.loads((RAIZ / "reports/validation_report.json").read_text(encoding="utf-8"))
resumen = pd.DataFrame([{
    "Archivo": a["archivo"],
    "Registros": a["registros"],
    "Errores": len(a["errores"]),
    "Advertencias": len(a["advertencias"]),
    "Aprobado": "Sí" if a["aprobado"] else "No",
    "SHA-256 (inicio)": a["sha256"][:16],
} for a in reporte_validacion["archivos"]])
print("Contrato:", reporte_validacion["contrato"], "| Reglas:", ", ".join(reporte_validacion["reglas"]))
resumen

Contrato: contrato_inferencia_iris v1.0.0 | Reglas: columnas, nulo, tipo, rango, dominio, relacion, clases_completas, sin_solapamiento_train_test


,Archivo,Registros,Errores,Advertencias,Aprobado,SHA-256 (inicio)
0,data/raw/iris.csv,150,0,1,Sí,403ce7582d1f16b2
1,data/processed/train.csv,120,0,1,Sí,99cf305e7ab13a55
2,data/processed/test.csv,30,0,0,Sí,90e5031cbe3fffb9


Los tres archivos pasan la validación. Solo se genera una advertencia por una flor duplicada que ya viene en el conjunto de datos Iris. Esta queda registrada en el log, pero no detiene el pipeline, ya que es un dato original y no un error al momento de ingresar la información.

## 14. Prueba del control de calidad: ¿qué pasa si llegan datos inválidos?

Primero se guarda una copia de `train.csv` y luego se modifican cuatro valores para introducir diferentes errores: un dato nulo, un texto, un valor fuera del rango permitido y una clase que no existe. Después se ejecutan la validación y el entrenamiento para comprobar cómo responde el pipeline ante estos problemas. Finalmente, se recupera el archivo original y se vuelve a ejecutar la validación.

In [21]:
ruta_train = RAIZ / "data/processed/train.csv"
original = ruta_train.read_text(encoding="utf-8")

dañado = pd.read_csv(ruta_train, dtype=str)
dañado.loc[2, "sepal_length"] = ""      # nulo
dañado.loc[4, "petal_length"] = "abc"   # tipo
dañado.loc[6, "petal_width"] = "9.5"    # rango
dañado.loc[8, "species"] = "4"          # dominio
dañado.to_csv(ruta_train, index=False)

print("Validación con datos dañados")
ejecutar("-m", "src.validate_data")
print("Entrenamiento con datos dañados")
ejecutar("-m", "src.train")

Validación con datos dañados
2026-11-24 17:47:00 | INFO    | validate  | APROBADO | data/raw/iris.csv | 150 registros | 0 errores  | 1 advertencias
2026-11-24 17:47:00 | WARNING | validate  |   Fila 144 duplica la fila 103 ['5.8', '2.7', '5.1', '1.9', '2']
2026-11-24 17:47:00 | INFO    | validate  | RECHAZADO | data/processed/train.csv | 120 registros | 4 errores {'nulo': 1, 'tipo': 1, 'rango': 1, 'dominio': 1} | 1 advertencias
2026-11-24 17:47:00 | ERROR   | validate  |   fila 4 | sepal_length | nulo | Valor nulo o vacío | valor=''
2026-11-24 17:47:00 | ERROR   | validate  |   fila 6 | petal_length | tipo | Se esperaba float y llegó str | valor='abc'
2026-11-24 17:47:00 | ERROR   | validate  |   fila 8 | petal_width | rango | Fuera del rango lógico [0.1, 2.5] cm | valor='9.5'
2026-11-24 17:47:00 | ERROR   | validate  |   fila 10 | species | dominio | Clase fuera de [0, 1, 2] | valor='4'
2026-11-24 17:47:00 | WARNING | validate  |   Fila 61 duplica la fila 48 ['5.8', '2.7', '5.1', '1.9

La validación identifica los cuatro errores e indica en cada caso la fila, el campo afectado, la regla incumplida y el valor encontrado. Como resultado, el proceso termina con código 1, indicando que los datos no fueron aprobados.

Al intentar ejecutar el entrenamiento, este revisa el reporte de validación y detecta que fue rechazado, por lo que no permite continuar. Si se ejecuta el pipeline completo, este error también impide que se realicen las etapas siguientes.

Finalmente, se restaura el archivo original y se ejecuta nuevamente la validación para comprobar que los datos estén correctos y dejar el reporte aprobado.

In [22]:
ruta_train.write_text(original, encoding="utf-8")
ejecutar("-m", "src.validate_data")

2026-11-24 17:47:10 | INFO    | validate  | APROBADO | data/raw/iris.csv | 150 registros | 0 errores  | 1 advertencias
2026-11-24 17:47:10 | WARNING | validate  |   Fila 144 duplica la fila 103 ['5.8', '2.7', '5.1', '1.9', '2']
2026-11-24 17:47:10 | INFO    | validate  | APROBADO | data/processed/train.csv | 120 registros | 0 errores  | 1 advertencias
2026-11-24 17:47:10 | WARNING | validate  |   Fila 61 duplica la fila 48 ['5.8', '2.7', '5.1', '1.9', '2']
2026-11-24 17:47:10 | INFO    | validate  | APROBADO | data/processed/test.csv | 30 registros | 0 errores  | 0 advertencias
2026-11-24 17:47:10 | INFO    | validate  | APROBADO | reglas de conjunto: clases_completas, sin_solapamiento_train_test
2026-11-24 17:47:10 | INFO    | validate  | Reporte guardado en reports\validation_report.json
2026-11-24 17:47:10 | INFO    | validate  | Ejemplo válido   -> aprobado
2026-11-24 17:47:10 | INFO    | validate  | Ejemplo inválido -> 4 errores detectados
2026-11-24 17:47:10 | INFO    | validate 

## 15. Contrato de datos en la inferencia

Se prueban los dos ejemplos definidos en el contrato. El ejemplo válido devuelve la especie predicha (versicolor), mientras que el inválido es rechazado antes de llegar al modelo y muestra los cuatro errores establecidos en el contrato.


In [23]:
ejecutar("-m", "src.predict", "--ejemplo", "valida")
ejecutar("-m", "src.predict", "--ejemplo", "invalida")

[
  {
    "prediction": "versicolor",
    "class_id": 1
  }
]

Código de salida: 0
[
  {
    "errores": [
      {
        "campo": "species",
        "regla": "columnas",
        "mensaje": "Campo no permitido en inferencia: es la variable objetivo (fuga de datos)",
        "valor": "0"
      },
      {
        "campo": "sepal_length",
        "regla": "tipo",
        "mensaje": "Se esperaba float y llegó str",
        "valor": "'5.1'"
      },
      {
        "campo": "sepal_width",
        "regla": "nulo",
        "mensaje": "Valor nulo o vacío",
        "valor": "None"
      },
      {
        "campo": "petal_length",
        "regla": "rango",
        "mensaje": "Fuera del rango lógico [1.0, 7.0] cm",
        "valor": "-1.4"
      }
    ]
  }
]

Código de salida: 0


También es posible validar varios registros a la vez mediante un archivo JSON, simulando la forma en que llegarían los datos a la API. En este ejemplo, la segunda flor tiene las medidas del pétalo y del sépalo intercambiadas, mientras que la tercera presenta un valor decimal con coma en lugar de punto, ingresado como texto.

In [24]:
solicitud = [
    {"sepal_length": 5.0, "sepal_width": 3.4, "petal_length": 1.5, "petal_width": 0.2},
    {"sepal_length": 4.5, "sepal_width": 3.0, "petal_length": 6.0, "petal_width": 2.0},
    {"sepal_length": "6,3", "sepal_width": 2.5, "petal_length": 5.0, "petal_width": 1.9},
]
(RAIZ / "reports/solicitud_ejemplo.json").write_text(json.dumps(solicitud, indent=2), encoding="utf-8")
ejecutar("-m", "src.predict", "--archivo", "reports/solicitud_ejemplo.json")

[
  {
    "prediction": "setosa",
    "class_id": 0
  },
  {
    "errores": [
      {
        "campo": "petal_length/sepal_length",
        "regla": "relacion",
        "mensaje": "En las 150 flores del dataset la longitud del pétalo es menor que la del sépalo. Un registro que no cumpla esta relación probablemente tiene las columnas intercambiadas.",
        "valor": "(6.0, 4.5)"
      }
    ]
  },
  {
    "errores": [
      {
        "campo": "sepal_length",
        "regla": "tipo",
        "mensaje": "Se esperaba float y llegó str",
        "valor": "'6,3'"
      }
    ]
  }
]

Código de salida: 0


La primera flor se clasifica como setosa, mientras que la segunda y la tercera son rechazadas con su respectivo error (relación entre campos y tipo de dato). Un registro con errores no impide que los demás se procesen.

## 16. Validación del modelo y comparación con la baseline

In [25]:
evaluacion = json.loads((RAIZ / "reports/evaluation_report.json").read_text(encoding="utf-8"))
tabla = pd.DataFrame(evaluacion["resultados"]).T[
    ["train_accuracy", "cv_accuracy_mean", "test_accuracy", "test_f1_macro", "test_errores", "brecha_train_test"]]
tabla.index = ["Baseline (100 árboles, sin límite)", "Seleccionado (120 árboles, profundidad 4)"]
tabla.columns = ["Accuracy entrenamiento", "CV media (train)", "Accuracy prueba", "F1 macro prueba",
                 "Errores prueba", "Brecha train–test"]
display(tabla.style.format("{:.3f}", subset=tabla.columns.drop("Errores prueba")))

controles = pd.Series(evaluacion["controles"]).map({True: "Cumple", False: "No cumple"})
controles.to_frame("Resultado")

,Accuracy entrenamiento,CV media (train),Accuracy prueba,F1 macro prueba,Errores prueba,Brecha train–test
"Baseline (100 árboles, sin límite)",1.000,0.950,0.900,0.900,3,0.100
"Seleccionado (120 árboles, profundidad 4)",1.000,0.950,0.967,0.967,1,0.033


,Resultado
accuracy_min,Cumple
f1_macro_min,Cumple
cv_accuracy_min,Cumple
no_inferior_al_baseline,Cumple
contrato_ejemplo_valido_predicho,Cumple
contrato_ejemplo_invalido_rechazado,Cumple


Los dos modelos obtienen el mismo resultado en la validación cruzada (0.950). Sin embargo, al evaluarlos con los datos de prueba, el modelo seleccionado se equivoca en 1 de las 30 flores (0.967), mientras que la baseline falla en 3 (0.900). Aunque la diferencia es pequeña, equivale a dos flores y permite comprobar que limitar la profundidad del modelo a 4 no reduce su rendimiento.

Con la nueva partición estratificada, el modelo seleccionado también clasifica correctamente todos los datos de entrenamiento, a diferencia de la Actividad 1.1, donde se eligió precisamente porque no memorizaba esos datos. Por esta razón, su elección se justifica ahora por los resultados en prueba, por su menor brecha entre entrenamiento y prueba (0.033 frente a 0.100 de la baseline) y por su menor complejidad.

En general, el modelo seleccionado cumple con todos los criterios de calidad establecidos. Además, con esta nueva partición ya no se obtiene el 100% de accuracy que se había alcanzado en la Actividad 1.1, lo cual coincide con lo explicado en clase, donde se plantea que un resultado perfecto debe revisarse con cuidado y no necesariamente interpretarse como una ventaja.

## 17. Juego "¡Rompe el pipeline!" visto en clase

Se realizan los siete sabotajes vistos en clase y en cada caso, se ejecuta el pipeline omitiendo la etapa de preparación de los datos (`prepare`). La tabla muestra en qué etapa se detiene el proceso y cuál es el primer error que queda registrado en el log. La ejecución completa tarda aproximadamente dos minutos.

Como comparación, en el ejercicio realizado en clase, los sabotajes 5 y 6 fueron detectados por las pruebas, mientras que el sabotaje 7 no fue identificado por ninguna etapa.

In [26]:
import re

ESCENARIOS = {
    1: "«cinco» en lugar de un número",
    2: "Celda vacía en petal_width",
    3: "Registro en milímetros",
    4: "Especie con código 3",
    5: "Sin flores virginica",
    6: "40 % de etiquetas erradas",
    7: "Prueba copiada en entrenamiento",
}
EN_CLASE = {1: "Validación", 2: "Validación", 3: "Validación", 4: "Validación",
            5: "Pruebas", 6: "Pruebas", 7: "Nadie"}

entorno = {**os.environ, "PYTHONUTF8": "1"}
juego = []
for escenario, descripcion in ESCENARIOS.items():
    subprocess.run([sys.executable, "romper_datos.py", str(escenario)], cwd=RAIZ, env=entorno,
                   capture_output=True)
    salida = subprocess.run([sys.executable, "run_pipeline.py", "--sin-preparar"], cwd=RAIZ, env=entorno,
                            capture_output=True, text=True, encoding="utf-8")
    log = salida.stdout + salida.stderr
    etapa = re.search(r"Etapa (\w+) falló", log)
    errores = [linea.split("|", 3)[-1].strip() for linea in log.splitlines() if "| ERROR" in linea]
    juego.append({"Sabotaje": f"{escenario}. {descripcion}", "En la clase": EN_CLASE[escenario],
                  "Etapa que lo detiene aquí": etapa.group(1) if etapa else "Nadie",
                  "Primer error en el log": errores[0] if errores else ""})

# Los datos quedaron saboteados por el último escenario, el pipeline completo los regenera limpios
restaurado = subprocess.run([sys.executable, "run_pipeline.py"], cwd=RAIZ, env=entorno, capture_output=True)
print("Datos restaurados con el pipeline completo. Código de salida:", restaurado.returncode)

pd.set_option("display.max_colwidth", 110)
pd.DataFrame(juego)

Datos restaurados con el pipeline completo. Código de salida: 0


,Sabotaje,En la clase,Etapa que lo detiene aquí,Primer error en el log
0,1. «cinco» en lugar de un número,Validación,validate,fila 2 | sepal_length | tipo | Se esperaba float y llegó str | valor='cinco'
1,2. Celda vacía en petal_width,Validación,validate,fila 2 | petal_width | nulo | Valor nulo o vacío | valor=''
2,3. Registro en milímetros,Validación,validate,"fila 3 | sepal_length | rango | Fuera del rango lógico [4.0, 8.0] cm | valor='49.0'"
3,4. Especie con código 3,Validación,validate,"fila 2 | species | dominio | Clase fuera de [0, 1, 2] | valor='3'"
4,5. Sin flores virginica,Pruebas,validate,train.csv | clases_completas | Faltan especies: ['virginica']
5,6. 40 % de etiquetas erradas,Pruebas,evaluate,Control accuracy_min FALLA
6,7. Prueba copiada en entrenamiento,Nadie,validate,train.csv/test.csv | sin_solapamiento_train_test | 30 de 30 flores de prueba también están en entrenamient...


Los sabotajes del 1 al 4 se detectan durante la validación, igual que en el ejercicio de clase. Además, los sabotajes 5 y 7 también se identifican en esta etapa gracias a las reglas que revisan el conjunto de datos, por lo que no llegan al entrenamiento. En clase, el sabotaje 5 solo se detectó en las pruebas, después de entrenar, y el 7 no lo detectó ninguna etapa.

El sabotaje 6 requiere una revisión diferente, ya que no es posible identificarlo directamente con los datos. Esto se debe a que una etiqueta incorrecta sigue teniendo un valor válido, como 0, 1 o 2. Por esta razón, se detecta más adelante, durante la evaluación del modelo (`evaluate`).

Incluso con el 40% de las etiquetas incorrectas, el modelo seleccionado todavía alcanza un accuracy de 0.867, que habría superado el umbral de 0.80 utilizado en la demostración de clase. Por esa razón en este proyecto se establece un mínimo de
0.90, buscando que el control de calidad sea más exigente.

## 18. Pruebas automatizadas

In [27]:
ejecutar("-m", "pytest", "-v", "--color=no")

============================= test session starts =============================
platform win32 -- Python 3.12.10, pytest-9.1.1, pluggy-1.7.0 -- d:\1. Esp. Ciencia de datos\2026-2\6. Seminario de grado 2\2. Segunda entrega\Repositorio_Entrega2\.venv\Scripts\python.exe
cachedir: .pytest_cache
rootdir: d:\1. Esp. Ciencia de datos\2026-2\6. Seminario de grado 2\2. Segunda entrega\Repositorio_Entrega2
configfile: pytest.ini
testpaths: tests
plugins: anyio-4.16.0, platformdirs-4.14.8
collecting ... collected 38 items

tests/test_model.py::test_particion_reproducible PASSED                  [  2%]
tests/test_model.py::test_columnas_coinciden_con_el_contrato PASSED      [  5%]
tests/test_model.py::test_validacion_de_datos_aprobada PASSED            [  7%]
tests/test_model.py::test_modelo_con_hiperparametros_del_modelo_seleccionado PASSED [ 10%]
tests/test_model.py::test_accuracy_supera_el_umbral PASSED               [ 13%]
tests/test_model.py::test_salida_de_inferencia_cumple_el_contrato PASSE

Las 38 pruebas se aprueban: 7 verifican los resultados del pipeline (partición, modelo, accuracy e inferencia) y 31 comprueban que el validador detecte cada tipo de error con la regla correcta.

## 19. Trazabilidad en MLflow

Cada vez que se ejecuta el pipeline se agregan dos nuevos runs al experimento `Proyecto_Final_Grado`, correspondientes a la baseline y al modelo seleccionado. Si también se ejecutó la sección 17, aparecen los registros adicionales del escenario 6, que es el único que alcanza la etapa de entrenamiento, y de la ejecución posterior en la que se restauran los datos originales.

La tabla resume los registros que quedaron guardados. Además, desde MLflow UI se pueden consultar los artefactos de cada ejecución, incluyendo el contrato, el reporte de validación, la configuración, el archivo `requirements.txt` y el modelo con su respectiva firma de entrada. El run del modelo seleccionado también incluye el reporte de evaluación.

In [28]:
mlflow.set_tracking_uri("sqlite:///mlflow.db")
runs = mlflow.search_runs(experiment_names=["Proyecto_Final_Grado"])
columnas = {
    "tags.mlflow.runName": "Run",
    "params.n_estimators": "Árboles",
    "params.max_depth": "Profundidad",
    "params.split_seed": "split_seed",
    "params.model_seed": "model_seed",
    "params.contrato_version": "Contrato",
    "metrics.cv_accuracy_mean": "CV media",
    "metrics.test_accuracy": "Accuracy prueba",
    "tags.validacion_modelo": "Validación modelo",
    "tags.sklearn_version": "scikit-learn",
    "tags.mlflow_version": "MLflow",
    "run_id": "Run ID",
}
runs[list(columnas)].rename(columns=columnas)

,Run,Árboles,Profundidad,split_seed,model_seed,Contrato,CV media,Accuracy prueba,Validación modelo,scikit-learn,MLflow,Run ID
0,Pipeline_Run_2_n120_depth4,120,4,42,42,1.0.0,0.950000,0.966667,aprobada,1.9.1,3.18.0,0d7547fabade4ac29bf7bc93dd4e689f
1,Pipeline_Baseline,100,None,42,42,1.0.0,0.950000,0.900000,referencia,1.9.1,3.18.0,052debff3ebc4e64b4bdd1cdedd74dcf
2,Pipeline_Run_2_n120_depth4,120,4,42,42,1.0.0,0.566667,0.866667,rechazada,1.9.1,3.18.0,1c5744e1156a469198a8b71af4493d7f
3,Pipeline_Baseline,100,None,42,42,1.0.0,0.516667,0.633333,referencia,1.9.1,3.18.0,e720313b1cdc4538b1af5cbe33155e4f
4,Pipeline_Run_2_n120_depth4,120,4,42,42,1.0.0,0.950000,0.966667,aprobada,1.9.1,3.18.0,71bcb176af3a434f8060d79da88d082d
5,Pipeline_Baseline,100,None,42,42,1.0.0,0.950000,0.900000,referencia,1.9.1,3.18.0,6e4119de67c4492383c74d85a9fa4808


La siguiente celda abre MLflow UI en segundo plano (`http://127.0.0.1:5000`). Los runs están en
**Experiments → Proyecto_Final_Grado**.

In [29]:
import time
import webbrowser

ui = subprocess.Popen([sys.executable, "-m", "mlflow", "ui", "--backend-store-uri", "sqlite:///mlflow.db",
                       "--host", "127.0.0.1", "--port", "5000"])
time.sleep(10)
webbrowser.open("http://127.0.0.1:5000")
print("MLflow UI disponible en http://127.0.0.1:5000")

MLflow UI disponible en http://127.0.0.1:5000


Esta celda cierra MLflow UI para liberar `mlflow.db`. Para dejarlo abierto y revisar los runs, se debe cambiar a `CERRAR_UI = False`.

In [30]:
CERRAR_UI = True

if CERRAR_UI:
    ui.terminate()
    print("MLflow UI cerrado.")
else:
    print("MLflow UI sigue activo.")

MLflow UI cerrado.


## 20. Conclusiones

- El trabajo realizado en la Actividad 1.1 ahora está organizado en un solo pipeline que cualquier persona puede ejecutar desde cero con `python run_pipeline.py --reiniciar` o `make reproducir`, obteniendo los mismos resultados en cada ejecución.

- La calidad de los datos y del modelo se revisa en dos momentos. Inicialmente, antes del entrenamiento, se verifican los tipos de datos, valores nulos, rangos, dominio, coherencia entre campos y que los conjuntos de entrenamiento y prueba estén completos y no se mezclen. Después, se evalúa el modelo con los umbrales mínimos establecidos y se compara su desempeño con la baseline. Si alguna de estas condiciones no se cumple, el proceso se detiene y deja registrado el motivo.

- En el ejercicio de sabotajes visto en clase, el pipeline logra detectar los siete escenarios, incluido el de los datos de prueba copiados en el entrenamiento, que durante la clase no fue identificado por ningún control. Sin embargo, esto no significa que se puedan detectar todos los errores, ya que un porcentaje menor de etiquetas incorrectas podría pasar las validaciones.

- El contrato de datos permite definir en un solo lugar las condiciones que debe cumplir la información que recibe el modelo. Estas reglas se utilizan tanto en la validación del entrenamiento como en la inferencia y la futura API, evitando diferencias entre los datos utilizados para entrenar y los que recibe el modelo cuando está en funcionamiento.

- La validación actual comprueba que los datos cumplan las reglas establecidas, pero no detecta cambios en su comportamiento a lo largo del tiempo. Para trabajar con datos reales, sería importante complementarla con el seguimiento de esos cambios (*data drift*) y definir los rangos permitidos con el apoyo de expertos en el tema, por ejemplo, un área de riesgo.